# Model F: PPE Safety Shoes Detection Training

**Architecture:** YOLOX-M  
**Classes:** `person`, `safety_shoes`, `no_safety_shoes` (3 classes)  
**Dataset:** ~3,772 images (SMALL dataset)  
**Input Size:** 640x640  

## Target Metrics

| Metric | Target | Min Acceptable |
|--------|--------|-----------|
| Precision | >= 0.88 | >= 0.82 |
| Recall | >= 0.85 | >= 0.80 |
| mAP@0.5 | >= 0.85 | >= 0.75 |
| FP Rate | < 4% | < 6% |
| FN Rate | < 5% | < 8% |

## Notes

- **Very small dataset** — only ~3.7K images, requiring aggressive augmentation
- 3-class detection similar to helmet model
- Safety shoes are small objects (lower portion of frame) — detection is inherently harder
- If this model fails at DG3, consider outsourcing (see dev plan Section 8.1.1)
- Training config: `features/ppe-shoes_detection/configs/02_training.yaml`
- Data config: `features/ppe-shoes_detection/configs/01_data.yaml`

In [ ]:
import sys
from pathlib import Path

# Add pipeline root to path so we can import our modules
PIPELINE_ROOT = Path("..").resolve()
sys.path.insert(0, str(PIPELINE_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from utils.config import load_config
from utils.device import get_device, set_seed

# Load configs
train_config = load_config(PIPELINE_ROOT / "configs" / "training" / "shoes.yaml")
data_config = load_config(PIPELINE_ROOT / "configs" / "data" / "shoes.yaml")

# Reproducibility
set_seed(train_config.get("seed", 42))
device = get_device()
print(f"Device: {device}")
print(f"Pipeline root: {PIPELINE_ROOT}")

## Data Preview

Load the safety shoes dataset and inspect class distribution.

In [ ]:
from data_transforms.dataset import YOLOXDataset, build_dataloader
from data_transforms.transforms import build_transforms

# Build dataset (no augmentation for preview)
preview_transforms = build_transforms(data_config, augment=False)
train_dataset = YOLOXDataset(
    config=data_config,
    split="train",
    transforms=preview_transforms,
)

print(f"Training samples: {len(train_dataset)}")
print(f"Classes: {train_dataset.class_names}")
print(f"Number of classes: {train_dataset.num_classes}")

if len(train_dataset) < 5000:
    print(f"\n[WARN] Small dataset ({len(train_dataset)} images). Aggressive augmentation is critical.")

In [ ]:
# Class distribution
class_counts = train_dataset.get_class_distribution()
print("Class distribution:")
for cls_name, count in class_counts.items():
    print(f"  {cls_name}: {count:,} instances")

# Plot distribution
colors = ["#3498db", "#2ecc71", "#e74c3c"]  # person=blue, safety_shoes=green, no_safety_shoes=red
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors)
ax.set_title("Safety Shoes Dataset — Class Distribution")
ax.set_ylabel("Instance Count")
for i, (cls, cnt) in enumerate(class_counts.items()):
    ax.text(i, cnt + 50, f"{cnt:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Display sample images with bounding boxes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
indices = np.random.choice(len(train_dataset), size=8, replace=False)

cls_colors = {0: "blue", 1: "green", 2: "red"}
cls_labels = {0: "person", 1: "safety_shoes", 2: "no_safety_shoes"}

for ax, idx in zip(axes.flat, indices):
    img, targets, _ = train_dataset[idx]
    if hasattr(img, "numpy"):
        img = img.numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1) if img.max() <= 1.0 else np.clip(img / 255.0, 0, 1)
    ax.imshow(img)
    for target in targets:
        cls_id, cx, cy, w, h = target[:5]
        color = cls_colors.get(int(cls_id), "yellow")
        x1, y1 = cx - w / 2, cy - h / 2
        rect = plt.Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        label = cls_labels.get(int(cls_id), str(int(cls_id)))
        ax.text(x1, y1 - 2, label, fontsize=6, color=color, weight="bold")
    ax.set_title(f"Sample #{idx}")
    ax.axis("off")

plt.suptitle("Safety Shoes Dataset — Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

## Data Augmentation Strategy for Small Datasets

With only ~3.7K images, data augmentation is critical to prevent overfitting.  
The training config should include these aggressive augmentation settings:

### Recommended augmentation strategy

1. **Mosaic** (enabled): Combines 4 images into one, effectively 4x the data diversity per batch
2. **Mixup** (higher ratio ~0.8): Blends two images together, regularizes the model
3. **Random affine**: Scale (0.5-1.5), rotation, shear for viewpoint variation
4. **Color jitter**: Aggressive HSV shifts to handle different lighting conditions
5. **Random horizontal flip**: Standard, since shoes appear on both sides
6. **Copy-paste augmentation**: If supported, paste shoe annotations onto different backgrounds

### Additional strategies to consider

- **Longer training**: More epochs (e.g., 500+) with early stopping to let the model see more augmented variations
- **Smaller learning rate**: Slower convergence helps with small datasets
- **Stronger weight decay**: Additional regularization against overfitting
- **Freeze backbone initially**: Train only the head for first N epochs, then unfreeze
- **Collect more factory data**: Even 500 additional labeled images from real cameras would help significantly

### Overfitting indicators to watch

- Training loss decreasing while validation loss increases
- Large gap between train mAP and val mAP (> 0.10)
- Val mAP plateauing early (< epoch 100)

## Training Configuration Review

Verify all hyperparameters — pay attention to augmentation settings for this small dataset.

In [ ]:
# Print training config summary
print("=" * 60)
print("TRAINING CONFIGURATION — Safety Shoes Detection")
print("=" * 60)
print(f"Model architecture : {train_config.get('arch', 'yolox-m')}")
print(f"Input size         : {train_config.get('input_size', 640)}")
print(f"Batch size         : {train_config.get('batch_size', 32)}")
print(f"Epochs             : {train_config.get('epochs', 500)}")
print(f"Learning rate      : {train_config.get('lr', 0.005)}")
print(f"LR scheduler       : {train_config.get('scheduler', 'cosine')}")
print(f"Warmup epochs      : {train_config.get('warmup_epochs', 10)}")
print(f"Weight decay       : {train_config.get('weight_decay', 5e-4)}")
print(f"FP16               : {train_config.get('fp16', True)}")
print(f"EMA                : {train_config.get('ema', True)}")
print(f"Early stopping     : {train_config.get('early_stopping', True)}")
print(f"Patience           : {train_config.get('patience', 50)}")
print("--- Augmentation (aggressive for small dataset) ---")
print(f"Mosaic             : {train_config.get('mosaic', True)}")
print(f"Mixup              : {train_config.get('mixup', 0.8)}")
print(f"HSV augmentation   : {train_config.get('hsv_aug', True)}")
print(f"Flip               : {train_config.get('flip', 0.5)}")
print(f"Scale range        : {train_config.get('scale', [0.5, 1.5])}")
print(f"Copy-paste         : {train_config.get('copy_paste', False)}")
print("---")
print(f"Pretrained weights : {train_config.get('pretrained', 'yolox_m.pth')}")
print(f"Num classes        : {data_config.get('num_classes', 3)}")
print(f"Output dir         : {train_config.get('output_dir', 'runs/shoes')}")
print("=" * 60)

# Warn if augmentation seems too mild for this dataset size
mixup = train_config.get("mixup", 0.5)
if mixup < 0.5:
    print("\n[WARN] Mixup ratio is low for a small dataset. Consider increasing to 0.8+.")

## Training

Launch YOLOX-M training on the safety shoes dataset.  
With only ~3.7K images, training should complete faster but watch for overfitting.

In [ ]:
from training.trainer import DetectionTrainer

# Create trainer with config
trainer = DetectionTrainer(
    config_path=PIPELINE_ROOT / "configs" / "training" / "shoes.yaml",
    overrides={
        "device": str(device),
    },
)

print(f"Trainer initialized: {trainer}")
print(f"Output directory: {trainer.output_dir}")
print(f"\nStarting training (small dataset — watch for overfitting)...")

# Train
history = trainer.train()

## Training Curves

Visualize loss and mAP progression. Watch for train/val divergence (overfitting).

In [ ]:
# Plot training curves with overfitting diagnostic
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve — watch for train/val divergence
axes[0].plot(history["epoch"], history["train_loss"], label="Train Loss", color="#e74c3c")
if "val_loss" in history:
    axes[0].plot(history["epoch"], history["val_loss"], label="Val Loss", color="#3498db")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss (watch for divergence)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# mAP curve
if "val_map50" in history:
    axes[1].plot(history["epoch"], history["val_map50"], label="mAP@0.5", color="#2ecc71")
    axes[1].axhline(y=0.85, color="red", linestyle="--", alpha=0.5, label="Target (0.85)")
    axes[1].axhline(y=0.75, color="orange", linestyle="--", alpha=0.5, label="Min (0.75)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mAP@0.5")
    axes[1].set_title("Validation mAP@0.5")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Learning rate schedule
if "lr" in history:
    axes[2].plot(history["epoch"], history["lr"], color="#9b59b6")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("LR Schedule")
    axes[2].grid(True, alpha=0.3)

plt.suptitle("Safety Shoes Detection — Training Curves", fontsize=14)
plt.tight_layout()
plt.show()

# Overfitting check
if "val_map50" in history and "train_loss" in history and "val_loss" in history:
    best_epoch = np.argmax(history["val_map50"])
    final_train_loss = history["train_loss"][-1]
    final_val_loss = history["val_loss"][-1]
    loss_gap = final_val_loss - final_train_loss
    print(f"\nBest epoch: {history['epoch'][best_epoch]}")
    print(f"Best mAP@0.5: {history['val_map50'][best_epoch]:.4f}")
    print(f"Final train/val loss gap: {loss_gap:.4f}")
    if loss_gap > 0.5:
        print("[WARN] Significant train/val loss gap — model may be overfitting.")

## Quick Evaluation

Load the best checkpoint and evaluate on the validation set.

In [ ]:
# Evaluate best model on validation set
metrics = trainer.evaluate(split="val")

print("=" * 60)
print("VALIDATION RESULTS — Safety Shoes Detection")
print("=" * 60)
print(f"mAP@0.5       : {metrics.get('map50', 0):.4f}  (target: >= 0.85)")
print(f"mAP@0.5:0.95  : {metrics.get('map50_95', 0):.4f}")
print(f"Precision      : {metrics.get('precision', 0):.4f}  (target: >= 0.88)")
print(f"Recall         : {metrics.get('recall', 0):.4f}  (target: >= 0.85)")
print("---")

# Per-class breakdown
if "per_class" in metrics:
    print("\nPer-class results:")
    for cls_name, cls_metrics in metrics["per_class"].items():
        print(f"  {cls_name}:")
        print(f"    AP@0.5     : {cls_metrics.get('ap50', 0):.4f}")
        print(f"    Precision  : {cls_metrics.get('precision', 0):.4f}")
        print(f"    Recall     : {cls_metrics.get('recall', 0):.4f}")

# Status check
map50 = metrics.get("map50", 0)
if map50 >= 0.85:
    print("\n[PASS] mAP@0.5 meets target. Proceed to export.")
elif map50 >= 0.75:
    print("\n[WARN] mAP@0.5 above minimum but below target.")
    print("  Consider: more factory data, stronger augmentation, or YOLOX-L escalation.")
else:
    print("\n[FAIL] mAP@0.5 below minimum.")
    print("  Recommended: collect more factory data, check label quality, or consider outsourcing.")

## Save & Next Steps

### Saved Artifacts

- **Best weights**: `runs/shoes/best.pth`
- **Last weights**: `runs/shoes/last.pth`
- **Training log**: `runs/shoes/train_log.json`
- **W&B run**: Check your W&B dashboard for full metrics and artifacts

### Next Steps

1. **If mAP >= 0.85**: Export to ONNX and proceed to edge testing
2. **If 0.75 <= mAP < 0.85**:
   - Run error analysis with FiftyOne — focus on small shoe bounding boxes
   - Collect more factory-specific data (even 500 images helps significantly)
   - Try YOLOX-L for a larger backbone
3. **If mAP < 0.75**:
   - Escalate to YOLOX-L
   - If YOLOX-L also fails at DG3, consider outsourcing this model (see dev plan Section 8.1.1)

### Small Dataset Remediation

If accuracy is insufficient, prioritize these actions:
1. Collect 500+ images from actual factory cameras
2. Use SAM3 for semi-automatic annotation to speed up labeling
3. Try test-time augmentation (TTA) during inference for accuracy boost

### Export Command

```bash
python core/p09_export/export.py --model runs/shoes/best.pt --training-config features/ppe-shoes_detection/configs/06_training.yaml
```